In [ ]:
import httpx

from rdakt_ai import RdaktMiddleware


def log_redactions(original_text, anonymized_text, mapping):
    print(f"Original:    {original_text}")
    print(f"Sent to LLM: {anonymized_text}")
    for real_value, token in mapping.items():
        print(f"  {real_value} -> {token}")


def log_restored(anonymized_text, restored_text):
    print(f"LLM saw:  {anonymized_text}")
    print(f"You see:  {restored_text}")


middleware = RdaktMiddleware(
    inner=httpx.AsyncHTTPTransport(),
    on_anonymized=log_redactions,
    on_deanonymized=log_restored,
)

In [16]:
from anthropic import AsyncAnthropic

client = AsyncAnthropic(http_client=httpx.AsyncClient(transport=middleware))

response = await client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Just repeat what i wrote after this message: Summarize the case for John Smith (SSN 123-45-6789)",
        }
    ],
)

Original:    Just repeat what i wrote after this message: Summarize the case for John Smith (SSN 123-45-6789)
Sent to LLM: Just repeat what i wrote after this message: Summarize the case for John Smith (SSN <SSN_1>)
  <SSN_1> -> 123-45-6789
LLM saw:  {"model":"claude-sonnet-4-20250514","id":"msg_0134dtEr5r5csCkenXi8Mtss","type":"message","role":"assistant","content":[{"type":"text","text":"Summarize the case for John Smith (SSN <SSN_1>)"}],"stop_reason":"end_turn","stop_sequence":null,"stop_details":null,"usage":{"input_tokens":34,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":21,"service_tier":"standard","inference_geo":"not_available"}}
You see:  {"model": "claude-sonnet-4-20250514", "id": "msg_0134dtEr5r5csCkenXi8Mtss", "type": "message", "role": "assistant", "content": [{"type": "text", "text": "Summarize the case for John Smith (SSN 123-45-6789)"}], "stop_reason": "end_turn"

In [17]:
print(response.content[0].text)

Summarize the case for John Smith (SSN 123-45-6789)
